In [2]:
import pandas as pd
from sentence_transformers import SentenceTransformer
from tqdm import tqdm
from models import TfIdfEmbedder, CountVectorizerEmbedder

tqdm.pandas()

C:\Users\mjantscher\PycharmProjects\clariah_summer_school\dse-ml-2026\materials\2026-09-22_tuesday\05_jantscher_embeddings\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### 1. Prepare parlamint dataset

#### 1.1. Load Dataset (Sentence-Wise)

In [3]:
# Load parlamint dataset
df_parlamint = pd.read_csv("../../datasets/parlamint/parlamint-it-is-2022.txt", sep="\t").head(10000)
df_parlamint_subset = df_parlamint.copy(deep=True).head(100)
df_parlamint

,ID,Parent_ID,Text
0,ParlaMint-IS_2022-01-17-20.seg2.1,ParlaMint-IS_2022-01-17-20.u1,President of the United States reports:
1,ParlaMint-IS_2022-01-17-20.seg3.1,ParlaMint-IS_2022-01-17-20.u1,"I have decided, according to the proposal of t..."
2,ParlaMint-IS_2022-01-17-20.seg4.1,ParlaMint-IS_2022-01-17-20.u1,"Arrange sites, January 11th, 2022."
3,ParlaMint-IS_2022-01-17-20.seg6.1,ParlaMint-IS_2022-01-17-20.u1,Katrín Jakobsdóttir's daughter.
4,ParlaMint-IS_2022-01-17-20.seg7.1,ParlaMint-IS_2022-01-17-20.u1,Presidential Letters for a meeting of the Gene...
...,...,...,...
9995,ParlaMint-IS_2022-01-27-28.seg113.4,ParlaMint-IS_2022-01-27-28.u58,"Some of these are good, but the big picture is..."
9996,ParlaMint-IS_2022-01-27-28.seg113.5,ParlaMint-IS_2022-01-27-28.u58,The variable was never taken out whether it wa...
9997,ParlaMint-IS_2022-01-27-28.seg114.1,ParlaMint-IS_2022-01-27-28.u58,I will not vote with this case.
9998,ParlaMint-IS_2022-01-27-28.seg114.2,ParlaMint-IS_2022-01-27-28.u58,I won't get in the way of this case.


#### 1.2. Group Dataset per Utterance

In [4]:
# Group sentence by utterance (=Parent_ID)
df_parlamint_grouped = (df_parlamint.groupby(["Parent_ID"])["Text"]
                        .apply(lambda s: " ".join(s))
                        .reset_index(name="utterance_text"))
print(f"Unique utterances: {df_parlamint_grouped.shape[0]}")
df_parlamint_grouped

Unique utterances: 660


,Parent_ID,utterance_text
0,ParlaMint-IS_2022-01-17-20.u1,President of the United States reports: I have...
1,ParlaMint-IS_2022-01-17-20.u10,"Before the weekend, an article by Stefánssonar..."
2,ParlaMint-IS_2022-01-17-20.u11,"I read this decision in Perconte, which is not..."
3,ParlaMint-IS_2022-01-17-20.u12,"In fact, this is shown in the letter quoted by..."
4,ParlaMint-IS_2022-01-17-20.u13,"Yes, that's right. That's right. A senator who..."
...,...,...
655,ParlaMint-IS_2022-01-27-28.u58,Here we vote for a case that involves massive ...
656,ParlaMint-IS_2022-01-27-28.u6,"I come up here to agree with this, this case i..."
657,ParlaMint-IS_2022-01-27-28.u7,In his article at Science yesterday and also i...
658,ParlaMint-IS_2022-01-27-28.u8,The president still reminds us of a limited ta...


In [5]:
sample_utterance = df_parlamint[df_parlamint["Parent_ID"] == "ParlaMint-IS_2022-01-17-20.u1"]["Text"]
sample_utterance

0              President of the United States reports:
1    I have decided, according to the proposal of t...
2                   Arrange sites, January 11th, 2022.
3                      Katrín Jakobsdóttir's daughter.
4    Presidential Letters for a meeting of the Gene...
5    I'd like to use this opportunity here after re...
Name: Text, dtype: str

### 2. Different Text Embedding Algorithms

#### 2.1. Count Vectorizer (Sparse) Embeddings

In [6]:
# Adding the whole parlamint dataset as vocabulary
# cv_model = CountVectorizerEmbedder(vocabulary=df_parlamint["Text"].to_list(), min_df=100, stop_words='english',
#                                    n_gram_range=(1, 3))

# Adding just the utterance sample as vocabulary
cv_model = CountVectorizerEmbedder(vocabulary=df_parlamint_grouped["utterance_text"], max_features=10,stop_words='english')

In [7]:
cv_embeddings = cv_model.embed(sample_utterance)
print(f"Number features: {len(cv_model.embedding_model.get_feature_names_out())}", cv_model.embedding_model.get_feature_names_out())
print(f"Shape embedding array: {cv_embeddings.toarray().shape}")
df_cv_output = pd.DataFrame(columns=cv_model.embedding_model.get_feature_names_out(), data=cv_embeddings.toarray())
df_cv_output

Call 'transform' only...
Number features: 10 ['council' 'government' 'health' 'just' 'like' 'minister' 'need' 'people'
 'think' 'time']
Shape embedding array: (6, 10)


,council,government,health,just,like,minister,need,people,think,time
0,0,0,0,0,0,0,0,0,0,0
1,1,0,0,0,0,1,0,0,0,0
2,0,0,0,0,0,0,0,0,0,0
3,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0
5,0,0,0,0,1,0,0,0,0,0


#### 2.2 TF-IDF (Sparse) Embeddings

In [8]:
# Adding the whole parlamint dataset as vocabulary
# tfidf_model = TfIdfEmbedder(vocabulary=df_parlamint["Text"].to_list(), min_df=100, stop_words='english')

# Adding just the utterance sample as vocabulary
tfidf_model = TfIdfEmbedder(vocabulary=df_parlamint_grouped["utterance_text"], max_features=10, stop_words='english')

In [9]:
tfidf_embeddings = tfidf_model.embed(sample_utterance)
print(f"Number features: {len(tfidf_model.embedding_model.get_feature_names_out())}", tfidf_model.embedding_model.get_feature_names_out())
print(f"Shape embedding array: {tfidf_embeddings.toarray().shape}")
df_tfidf_output = pd.DataFrame(columns=tfidf_model.embedding_model.get_feature_names_out(), data=tfidf_embeddings.toarray())
df_tfidf_output

Call 'transform' only...
Number features: 10 ['council' 'government' 'health' 'just' 'like' 'minister' 'need' 'people'
 'think' 'time']
Shape embedding array: (6, 10)


,council,government,health,just,like,minister,need,people,think,time
0,0.000000,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0
1,0.735172,0.0,0.0,0.0,0.0,0.677881,0.0,0.0,0.0,0.0
2,0.000000,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0
3,0.000000,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0
4,0.000000,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0
5,0.000000,0.0,0.0,0.0,1.0,0.000000,0.0,0.0,0.0,0.0


#### 2.3 Sentence Transformer (Dense) Embeddings

In [10]:
st_model_small = SentenceTransformer('all-minilm-l6-v2')

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6810.76it/s]


In [13]:
# Encode sentence-wise
# sample_utterance is a pandas Series; encode expects a string or a list of strings.
st_embeddings = st_model_small.encode(sample_utterance.tolist())
print(f"Number of sentence embeddings: {len(st_embeddings)}")
print(f"Shape embedding array: {st_embeddings.shape}")
st_embeddings

Number of sentence embeddings: 6
Shape embedding array: (6, 384)


array([[ 0.00364318,  0.00757526, -0.01352413, ..., -0.01752407,
         0.00990046,  0.0602178 ],
       [-0.04760472, -0.0689887 ,  0.02710492, ..., -0.06230189,
        -0.14385444,  0.03971655],
       [-0.01698013, -0.04804507, -0.01744974, ..., -0.03905816,
        -0.07595795,  0.00352386],
       [-0.083314  , -0.04858719,  0.00143911, ...,  0.0383972 ,
         0.06088393, -0.01607141],
       [-0.07626821, -0.06527785,  0.06610002, ...,  0.02295849,
        -0.1082601 , -0.0659192 ],
       [ 0.01009848, -0.04537036,  0.04564783, ..., -0.00065688,
        -0.08547057, -0.00425589]], shape=(6, 384), dtype=float32)

In [14]:
# Encode utterance-wise
st_embeddings_u = st_model_small.encode(" ".join(sample_utterance))
print(f"Number features: {len(st_embeddings_u)}")
print(f"Shape embedding array: {st_embeddings_u.shape}")
st_embeddings_u

Number features: 384
Shape embedding array: (384,)


array([-5.19020036e-02, -1.02959491e-01,  6.31395727e-02,  2.98001692e-02,
       -2.51417160e-02, -9.66067053e-03, -7.38494396e-02, -2.23561749e-02,
       -6.58514872e-02,  7.95470644e-03, -6.14716969e-02,  9.11197811e-03,
       -7.41380900e-02, -8.91723949e-03,  4.29925248e-02,  3.98151390e-02,
        5.71959419e-04, -2.14750133e-02,  4.03534845e-02, -3.10853170e-03,
        4.81390879e-02,  3.13877687e-02,  2.43080799e-02,  3.66794015e-03,
       -4.10232060e-02, -1.00540509e-02, -1.73179470e-02, -3.86848152e-02,
       -1.10482723e-02,  6.46823198e-02,  5.51152080e-02, -5.29566640e-03,
        7.86908492e-02,  2.21064128e-02,  5.85365370e-02,  2.10549473e-03,
        5.57183772e-02,  3.54420543e-02,  3.88753861e-02, -5.69795966e-02,
       -1.24397492e-02, -6.79901391e-02,  2.59146076e-02, -4.62712580e-03,
       -3.93818729e-02,  4.40331399e-02, -3.18672545e-02, -2.18536612e-03,
       -3.43882591e-02,  6.45774677e-02,  4.68761893e-03,  1.60148402e-03,
        1.24297496e-02, -

### 3. Encode whole Parlamint Dataset

#### 3.1 Encode with Sentence Transformer

In [15]:
# Encode utterance-wise dataset
df_parlamint_embeddings_per_utterance = st_model_small.encode(df_parlamint_grouped["utterance_text"].to_list(),
                                                     show_progress_bar=True)

# Encode sentence-wise dataset
df_parlamint_embeddings_per_sentence = st_model_small.encode(df_parlamint["Text"].to_list(), show_progress_bar=True)

Batches: 100%|██████████| 313/313 [00:26<00:00, 11.86it/s]


In [16]:
df_parlamint_grouped["embedding"] = list(df_parlamint_embeddings_per_utterance)
df_parlamint_grouped

,Parent_ID,utterance_text,embedding
0,ParlaMint-IS_2022-01-17-20.u1,President of the United States reports: I have...,"[-0.05190203, -0.10295953, 0.063139565, 0.0298..."
1,ParlaMint-IS_2022-01-17-20.u10,"Before the weekend, an article by Stefánssonar...","[-0.1046115, 0.06670189, -0.05656963, -0.02427..."
2,ParlaMint-IS_2022-01-17-20.u11,"I read this decision in Perconte, which is not...","[-0.0037421975, 0.07299823, -0.017051924, -0.0..."
3,ParlaMint-IS_2022-01-17-20.u12,"In fact, this is shown in the letter quoted by...","[-0.106303796, 0.063141495, -0.014823463, 0.01..."
4,ParlaMint-IS_2022-01-17-20.u13,"Yes, that's right. That's right. A senator who...","[-0.03764082, 0.10380773, -0.061545692, 0.0093..."
...,...,...,...
655,ParlaMint-IS_2022-01-27-28.u58,Here we vote for a case that involves massive ...,"[-0.01670022, -0.07529932, 0.02973297, 0.00316..."
656,ParlaMint-IS_2022-01-27-28.u6,"I come up here to agree with this, this case i...","[-0.004680503, 0.057901654, -0.0035116489, -0...."
657,ParlaMint-IS_2022-01-27-28.u7,In his article at Science yesterday and also i...,"[-0.026386984, 0.055420764, 0.045418244, 0.020..."
658,ParlaMint-IS_2022-01-27-28.u8,The president still reminds us of a limited ta...,"[0.017418459, -0.035378, 0.09752449, -0.021729..."


In [17]:
df_parlamint["embedding"] = list(df_parlamint_embeddings_per_sentence)
df_parlamint

,ID,Parent_ID,Text,embedding
0,ParlaMint-IS_2022-01-17-20.seg2.1,ParlaMint-IS_2022-01-17-20.u1,President of the United States reports:,"[0.0036431612, 0.0075753108, -0.013524168, 0.0..."
1,ParlaMint-IS_2022-01-17-20.seg3.1,ParlaMint-IS_2022-01-17-20.u1,"I have decided, according to the proposal of t...","[-0.04760472, -0.0689887, 0.02710492, 0.042773..."
2,ParlaMint-IS_2022-01-17-20.seg4.1,ParlaMint-IS_2022-01-17-20.u1,"Arrange sites, January 11th, 2022.","[-0.01698008, -0.04804512, -0.01744969, 0.0109..."
3,ParlaMint-IS_2022-01-17-20.seg6.1,ParlaMint-IS_2022-01-17-20.u1,Katrín Jakobsdóttir's daughter.,"[-0.083313994, -0.048587218, 0.0014390972, -0...."
4,ParlaMint-IS_2022-01-17-20.seg7.1,ParlaMint-IS_2022-01-17-20.u1,Presidential Letters for a meeting of the Gene...,"[-0.076268196, -0.06527783, 0.066100046, 0.010..."
...,...,...,...,...
9995,ParlaMint-IS_2022-01-27-28.seg113.4,ParlaMint-IS_2022-01-27-28.u58,"Some of these are good, but the big picture is...","[-0.010372347, -0.04594698, -0.015732476, -0.0..."
9996,ParlaMint-IS_2022-01-27-28.seg113.5,ParlaMint-IS_2022-01-27-28.u58,The variable was never taken out whether it wa...,"[0.0029584675, 0.07198213, -0.03759543, 0.0680..."
9997,ParlaMint-IS_2022-01-27-28.seg114.1,ParlaMint-IS_2022-01-27-28.u58,I will not vote with this case.,"[-0.00030057642, 0.038990874, 0.01706963, -0.0..."
9998,ParlaMint-IS_2022-01-27-28.seg114.2,ParlaMint-IS_2022-01-27-28.u58,I won't get in the way of this case.,"[-0.035732646, 0.06427998, 0.012762965, 0.0055..."


#### 3.2 Save output to pickle file

In [18]:
df_parlamint.to_pickle("df_parlamint_all-MiniLM-L6-v2.pkl")

#### 3.3 Encode Dataset with TF-IDF

In [19]:
# Adding the whole parlamint dataset as vocabulary
tfidf_model = TfIdfEmbedder(vocabulary=df_parlamint["Text"].to_list(), max_features=100, stop_words='english')

# Encode sentence-wise dataset
tfidf_embeddings_per_sentence = tfidf_model.embed(df_parlamint["Text"].to_list())

Call 'transform' only...


In [20]:
print(f"Number features: {len(tfidf_model.embedding_model.get_feature_names_out())}", tfidf_model.embedding_model.get_feature_names_out())
print(f"Shape embedding array: {tfidf_embeddings_per_sentence.toarray().shape}")
tfidf_embeddings_per_sentence.toarray()

Number features: 100 ['000' 'able' 'agree' 'ask' 'believe' 'better' 'business' 'care' 'case'
 'change' 'changes' 'children' 'clear' 'come' 'committee' 'community'
 'companies' 'council' 'country' 'course' 'decision' 'discussion' 'does'
 'don' 'economic' 'epidemic' 'especially' 'example' 'fact' 'financial'
 'general' 'going' 'good' 'government' 'health' 'highest' 'hospital'
 'iceland' 'important' 'increase' 'just' 'know' 'land' 'law' 'laws' 'like'
 'long' 'look' 'lot' 'make' 'management' 'matter' 'measures' 'members'
 'mental' 'minister' 'ministers' 'ministry' 'money' 'national' 'need'
 'new' 'number' 'order' 'pay' 'people' 'place' 'point' 'possible'
 'president' 'problem' 'public' 'really' 'report' 'right' 'rights' 'said'
 'say' 'security' 'senator' 'service' 'situation' 'social' 'society'
 'state' 'support' 'taken' 'talk' 'thank' 'things' 'think' 'time' 'today'
 'use' 've' 'want' 'way' 'work' 'year' 'years']
Shape embedding array: (10000, 100)


array([[0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       ...,
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.]], shape=(10000, 100))

In [34]:
df_parlamint["embedding"] = list(tfidf_embeddings_per_sentence.toarray())
df_parlamint

,ID,Parent_ID,Text,embedding
0,ParlaMint-IS_2022-01-17-20.seg2.1,ParlaMint-IS_2022-01-17-20.u1,President of the United States reports:,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
1,ParlaMint-IS_2022-01-17-20.seg3.1,ParlaMint-IS_2022-01-17-20.u1,"I have decided, according to the proposal of t...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
2,ParlaMint-IS_2022-01-17-20.seg4.1,ParlaMint-IS_2022-01-17-20.u1,"Arrange sites, January 11th, 2022.","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
3,ParlaMint-IS_2022-01-17-20.seg6.1,ParlaMint-IS_2022-01-17-20.u1,Katrín Jakobsdóttir's daughter.,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
4,ParlaMint-IS_2022-01-17-20.seg7.1,ParlaMint-IS_2022-01-17-20.u1,Presidential Letters for a meeting of the Gene...,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
...,...,...,...,...
9995,ParlaMint-IS_2022-01-27-28.seg113.4,ParlaMint-IS_2022-01-27-28.u58,"Some of these are good, but the big picture is...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
9996,ParlaMint-IS_2022-01-27-28.seg113.5,ParlaMint-IS_2022-01-27-28.u58,The variable was never taken out whether it wa...,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
9997,ParlaMint-IS_2022-01-27-28.seg114.1,ParlaMint-IS_2022-01-27-28.u58,I will not vote with this case.,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, ..."
9998,ParlaMint-IS_2022-01-27-28.seg114.2,ParlaMint-IS_2022-01-27-28.u58,I won't get in the way of this case.,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.719..."


#### 3.4 Save output to pickle file

In [22]:
df_parlamint.to_pickle("df_parlamint_all-tfidf.pkl")

#### 3.5 Load data from pickle file

In [40]:
# df_read_parlamint = pd.read_pickle("<filename_path>.pkl")
df_parlamint = pd.read_pickle("df_parlamint_all-MiniLM-L6-v2.pkl")

### 4. Calculate similarities between embeddings

In [23]:
from sentence_transformers import SentenceTransformer, util
from sklearn.metrics.pairwise import cosine_similarity

st_model_small = SentenceTransformer('all-minilm-l6-v2')

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6605.30it/s]


In [24]:
# 1. Example data
sentences = [
    "I deposited my paycheck at the bank yesterday.",
    "We had a picnic on the bank of the river.",
    "The financial institution announced a new savings account plan.",
    "She withdrew cash from the nearest ATM.",
    "The kids played near the riverbank after school."
]

query = "financial services"

# 2. Sentence Transformers embeddings
dense_embeddings_sentences = st_model_small.encode(sentences, convert_to_tensor=False)
dense_embeddings_query = st_model_small.encode([query], convert_to_tensor=False)
dense_similarities = util.cos_sim(dense_embeddings_query, dense_embeddings_sentences)[0].cpu().numpy()

# 3. TF-IDF embeddings
tfidf_model = TfIdfEmbedder(vocabulary=sentences, max_features=10, stop_words='english')
tfidf_embeddings_sentences = tfidf_model.embed(sentences)
tfidf_embeddings_query = tfidf_model.embed([query])
tfidf_similarities = cosine_similarity(tfidf_embeddings_query, tfidf_embeddings_sentences).flatten()

# 4. Compare rankings
df = pd.DataFrame({
    "sentence": sentences,
    "tfidf_similarity": tfidf_similarities,
    "st_similarity": dense_similarities
})
# df.sort_values(by=["st_similarity"], ascending=False, inplace=True)
df

Call 'transform' only...
Call 'transform' only...


,sentence,tfidf_similarity,st_similarity
0,I deposited my paycheck at the bank yesterday.,0.0,0.243794
1,We had a picnic on the bank of the river.,0.0,0.047487
2,The financial institution announced a new savi...,0.5,0.322175
3,She withdrew cash from the nearest ATM.,0.0,0.237477
4,The kids played near the riverbank after school.,0.0,0.080024


### 5. How to build a Simple QA System

#### 5.1 Get the Most Likely Utterance

In [42]:
import numpy as np
from sentence_transformers import util

# Given question
question = "What is the government policy on climate change?"
# question = "What about president of america?"

# 1. Embed the question
question_embedding = st_model_small.encode(question)

# 2. Compute cosine similarities
cosine_similarities = util.cos_sim(question_embedding, df_parlamint["embedding"])[0].cpu().numpy()

# 3. Get the index of the most similar utterance
most_similar_idx = int(np.argmax(cosine_similarities))

# 4. Retrieve the most similar text
most_similar_text = df_parlamint.iloc[most_similar_idx]["Text"]
# most_similar_text
print(f"Score: {cosine_similarities[most_similar_idx]:.4f} | Utterance: {most_similar_text}\n")

Score: 0.6772 | Utterance: So we're going to make it even. by the Minister's implementation of the government's strategy for climate change and by the process of climate management in its new formulation, public analysis of measures and other government policies.



In [41]:
len(df_parlamint["embedding"].iloc[1])

384

In [ ]:
df_parlamint["Text"]

#### 5.2 Get the Top-K relevant Utterances

In [43]:
question = "What is the government policy on climate change?"
# question = "America?"
k = 5  # choose how many results you want

# 1. Embed the question
question_embedding = st_model_small.encode(question)

# 2. Compute cosine similarities
cosine_similarities = util.cos_sim(question_embedding, df_parlamint["embedding"])[0].cpu().numpy()

# 3. Get indices of top-k most similar utterances
top_k_idx = np.argsort(cosine_similarities)[::-1][:k]

# 4. Retrieve the top-k utterances and their similarity scores
for idx in top_k_idx:
    text = df_parlamint.iloc[idx]["Text"]
    score = cosine_similarities[idx]
    print(f"Score: {score:.4f} | Utterance: {text}\n")


Score: 0.6772 | Utterance: So we're going to make it even. by the Minister's implementation of the government's strategy for climate change and by the process of climate management in its new formulation, public analysis of measures and other government policies.

Score: 0.6287 | Utterance: Is it consistent with the left-green climate policy?

Score: 0.6216 | Utterance: Does Ministers believe that a silicar's restarting can harmonize with the outlook and policy of the nation on climate?

Score: 0.6040 | Utterance: There is an emergency, Mrs. President, on climate issues, and this bill responds to that crisis by suggesting changes that will enable governments to be more effective and that will simply require them to face the climate crisis, the most important solution we are facing.

Score: 0.5818 | Utterance: In the 3rd. The bill suggests that the government's action programme on climate matters will be much more extensive, a better estimate of the estimated costs and the assessment of

In [ ]:
tfidf_model = TfIdfEmbedder(vocabulary=df_parlamint["Text"], max_features=1000, stop_words='english')
tfidf_embeddings_sentences = tfidf_model.embed(df_parlamint["Text"].to_list())

In [ ]:
df_parlamint["embedding"] = list(tfidf_embeddings_sentences.toarray())

In [ ]:
question = "What is the government policy on climate change?"
# question = "America?"
k = 5  # choose how many results you want

# 1. Embed the question
#question_embedding = tfidf_model.encode(question)
question_embedding = tfidf_model.embed([question])

# 2. Compute cosine similarities
cosine_similarities = util.cos_sim(question_embedding.toarray(), df_parlamint["embedding"])[0].cpu().numpy()

# 3. Get indices of top-k most similar utterances
top_k_idx = np.argsort(cosine_similarities)[::-1][:k]

# 4. Retrieve the top-k utterances and their similarity scores
for idx in top_k_idx:
    text = df_parlamint.iloc[idx]["Text"]
    score = cosine_similarities[idx]
    print(f"Score: {score:.4f} | Utterance: {text}\n")


In [45]:
import numpy as np
np.where(question_embedding.toarray() > 0)


AttributeError: 'numpy.ndarray' object has no attribute 'toarray'

In [44]:
print(f"Number features: {len(tfidf_model.embedding_model.get_feature_names_out())}", tfidf_model.embedding_model.get_feature_names_out())
print(f"Shape embedding array: {tfidf_embeddings.toarray().shape}")
df_tfidf_output = pd.DataFrame(columns=tfidf_model.embedding_model.get_feature_names_out(), data=tfidf_embeddings.toarray())

Number features: 10 ['account' 'announced' 'atm' 'bank' 'cash' 'deposited' 'financial'
 'institution' 'kids' 'near']
Shape embedding array: (6, 10)
